In [1]:
import boto3
import time
import pandas as pd

athena = boto3.client("athena", region_name="ap-southeast-2")

DATABASE = "pl_analytics"
OUTPUT_LOCATION = "s3://pl-analytics-2026-athena-results/"

In [2]:
def run_athena_query(query, database=DATABASE, output_location=OUTPUT_LOCATION):
    response = athena.start_query_execution(
        QueryString=query,
        QueryExecutionContext={"Database": database},
        ResultConfiguration={"OutputLocation": output_location},
    )
    query_execution_id = response["QueryExecutionId"]

    # Poll until the query finishes
    while True:
        status = athena.get_query_execution(QueryExecutionId=query_execution_id)
        state = status["QueryExecution"]["Status"]["State"]
        if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(1)

    if state != "SUCCEEDED":
        reason = status["QueryExecution"]["Status"].get("StateChangeReason", "Unknown error")
        raise RuntimeError(f"Query {state}: {reason}")

    # Athena writes results as a CSV in S3 — read it back directly
    result_path = f"{output_location}{query_execution_id}.csv"
    return pd.read_csv(result_path)

In [3]:
df = run_athena_query("""
    SELECT
    team,
    SUM(goals_scored) AS total_goals
FROM pl_analytics.match_features
GROUP BY team
ORDER BY total_goals DESC
LIMIT 10;
""")
df

,team,total_goals
0,Manchester City FC,77
1,Arsenal FC,71
2,Manchester United FC,69
3,Liverpool FC,63
4,Chelsea FC,58
5,AFC Bournemouth,58
6,Aston Villa FC,56
7,Brentford FC,55
8,Newcastle United FC,53
9,Brighton & Hove Albion FC,52


In [4]:
df = run_athena_query("""
    SELECT
    team,
    SUM(goals_conceded) AS total_conceded
FROM pl_analytics.match_features
GROUP BY team
ORDER BY total_conceded ASC
LIMIT 10;
""")
df

,team,total_conceded
0,Arsenal FC,27
1,Manchester City FC,35
2,Brighton & Hove Albion FC,46
3,Sunderland AFC,48
4,Aston Villa FC,49
5,Everton FC,50
6,Manchester United FC,50
7,Crystal Palace FC,51
8,Fulham FC,51
9,Nottingham Forest FC,51


In [5]:
df = run_athena_query("""
    SELECT
    ROUND(100.0 * SUM(CASE WHEN winner = 'HOME_TEAM' THEN 1 ELSE 0 END) / COUNT(*), 2) AS home_win_pct,
    ROUND(100.0 * SUM(CASE WHEN winner = 'AWAY_TEAM' THEN 1 ELSE 0 END) / COUNT(*), 2) AS away_win_pct,
    ROUND(100.0 * SUM(CASE WHEN winner = 'DRAW' THEN 1 ELSE 0 END) / COUNT(*), 2) AS draw_pct
FROM pl_analytics.clean_matches;
""")
df

,home_win_pct,away_win_pct,draw_pct
0,6.09,4.29,3.91


In [6]:
df = run_athena_query("""
    SELECT
    home_team_name,
    away_team_name,
    home_goals,
    away_goals,
    (home_goals + away_goals) AS total_goals
FROM pl_analytics.clean_matches
ORDER BY total_goals DESC
LIMIT 10;
""")
df

,home_team_name,away_team_name,home_goals,away_goals,total_goals
0,Fulham FC,Manchester City FC,4,5,9
1,Manchester United FC,AFC Bournemouth,4,4,8
2,Brighton & Hove Albion FC,Aston Villa FC,3,4,7
3,Newcastle United FC,Leeds United FC,4,3,7
4,Aston Villa FC,Sunderland AFC,4,3,7
5,Liverpool FC,West Ham United FC,5,2,7
6,Burnley FC,Brentford FC,3,4,7
7,Manchester United FC,Brighton & Hove Albion FC,4,2,6
8,Manchester City FC,Burnley FC,5,1,6
9,Crystal Palace FC,AFC Bournemouth,3,3,6


In [7]:
df = run_athena_query("""
    SELECT
    team,
    SUM(points) AS total_points,
    COUNT(*) AS played,
    SUM(CASE WHEN result = 'WIN' THEN 1 ELSE 0 END) AS wins,
    SUM(CASE WHEN result = 'DRAW' THEN 1 ELSE 0 END) AS draws,
    SUM(CASE WHEN result = 'LOSS' THEN 1 ELSE 0 END) AS losses
FROM pl_analytics.match_features
GROUP BY team
ORDER BY total_points DESC;
""")
df

,team,total_points,played,wins,draws,losses
0,Arsenal FC,85.0,38,26,7,5
1,Manchester City FC,78.0,38,23,9,6
2,Manchester United FC,71.0,38,20,11,7
3,Aston Villa FC,65.0,38,19,8,11
4,Liverpool FC,60.0,38,17,9,12
5,AFC Bournemouth,57.0,38,13,18,7
6,Sunderland AFC,54.0,38,14,12,12
7,Brentford FC,53.0,38,14,11,13
8,Brighton & Hove Albion FC,53.0,38,14,11,13
9,Chelsea FC,52.0,38,14,10,14


In [8]:
df = run_athena_query("""
    SELECT
    team,
    ROUND(AVG(rolling_form_pts), 2) AS avg_form,
    ROUND(AVG(avg_goals_scored), 2) AS avg_goals_scored,
    ROUND(AVG(avg_goals_conceded), 2) AS avg_goals_conceded
FROM pl_analytics.match_features
WHERE rolling_form_pts IS NOT NULL
GROUP BY team
ORDER BY avg_form DESC
LIMIT 10;
""")
df

,team,avg_form,avg_goals_scored,avg_goals_conceded
0,Arsenal FC,10.27,1.93,0.57
1,Manchester City FC,10.00,2.14,0.91
2,Manchester United FC,8.54,1.58,1.46
3,Aston Villa FC,7.97,1.17,1.05
4,Liverpool FC,7.81,1.81,1.40
5,AFC Bournemouth,7.03,1.56,1.65
6,Brighton & Hove Albion FC,6.84,1.39,1.33
7,Brentford FC,6.76,1.44,1.52
8,Chelsea FC,6.62,1.74,1.06
9,Everton FC,6.49,1.12,1.13


In [9]:
df = run_athena_query("""
    SELECT
    team,
    MAX(win_streak) AS longest_win_streak
FROM pl_analytics.match_features
GROUP BY team
ORDER BY longest_win_streak DESC
LIMIT 10;
""")
df

,team,longest_win_streak
0,Aston Villa FC,8
1,'startDate': '2023-08-11',8
2,'startDate': '2024-08-16',7
3,Manchester City FC,6
4,Liverpool FC,5
5,Arsenal FC,5
6,'startDate': '2025-08-15',5
7,Chelsea FC,4
8,Manchester United FC,4
9,AFC Bournemouth,3


In [10]:
df = run_athena_query("""
    SELECT
    team,
    ROUND(MAX(home_win_pct), 2) AS final_home_win_pct,
    ROUND(MAX(away_win_pct), 2) AS final_away_win_pct,
    ROUND(MAX(home_win_pct) - MAX(away_win_pct), 2) AS home_away_gap
FROM pl_analytics.match_features
GROUP BY team
ORDER BY home_away_gap DESC
LIMIT 10;
""")
df

,team,final_home_win_pct,final_away_win_pct,home_away_gap
0,'startDate': '2023-08-11',8.0,1.00,7.00
1,'startDate': '2024-08-16',7.0,1.00,6.00
2,'startDate': '2025-08-15',5.0,1.00,4.00
3,Burnley FC,1.0,0.20,0.80
4,Leeds United FC,1.0,0.33,0.67
5,Nottingham Forest FC,1.0,0.39,0.61
6,Sunderland AFC,1.0,0.40,0.60
7,Brentford FC,1.0,0.43,0.57
8,Everton FC,1.0,0.50,0.50
9,AFC Bournemouth,1.0,0.50,0.50


In [11]:
df = run_athena_query("""
    SELECT
    team,
    SUM(clean_sheet) AS total_clean_sheets
FROM pl_analytics.match_features
GROUP BY team
ORDER BY total_clean_sheets DESC
LIMIT 10;
""")
df

,team,total_clean_sheets
0,'startDate': '2023-08-11',1060
1,'startDate': '2024-08-16',956
2,'startDate': '2025-08-15',908
3,Arsenal FC,19
4,Manchester City FC,16
5,Crystal Palace FC,12
6,AFC Bournemouth,11
7,Sunderland AFC,11
8,Everton FC,11
9,Brentford FC,10
